In [6]:
# Feature Engineering and Model Training for Stock Sentiment Analysis
# ================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# Machine Learning imports
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor

# XGBoost and LightGBM
import xgboost as xgb
import lightgbm as lgb

# Visualization
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

print("All libraries imported successfully!")

# Set up paths
project_root = r"C:\Users\PC\OneDrive\Documents\DS project"
processed_data_dir = os.path.join(project_root, "processed_data")
models_dir = os.path.join(project_root, "models")
results_dir = os.path.join(project_root, "results")

# Create directories if they don't exist
os.makedirs(models_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

print(f"Project root: {project_root}")
print(f"Processed data: {processed_data_dir}")
print(f"Models: {models_dir}")
print(f"Results: {results_dir}")


All libraries imported successfully!
Project root: C:\Users\PC\OneDrive\Documents\DS project
Processed data: C:\Users\PC\OneDrive\Documents\DS project\processed_data
Models: C:\Users\PC\OneDrive\Documents\DS project\models
Results: C:\Users\PC\OneDrive\Documents\DS project\results


In [7]:

print("Loading cleaned merged data...")

# Load the cleaned merged dataset
data_path = os.path.join(processed_data_dir, "final_cleaned_data.csv")
df = pd.read_csv(data_path)

print(f"Data loaded successfully!")
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

# Convert Datetime to datetime type
df['Datetime'] = pd.to_datetime(df['Datetime'])
df = df.sort_values(['Stock', 'Datetime']).reset_index(drop=True)

# Display basic info
print(f"\nData info:")
print(f"   - Date range: {df['Datetime'].min()} to {df['Datetime'].max()}")
print(f"   - Stocks: {df['Stock'].unique()}")
print(f"   - Total samples: {len(df)}")

# Show sample data
print(f"\n Sample data:")
print(df.head())

Loading cleaned merged data...
Data loaded successfully!
Dataset shape: (9968, 32)
Columns: ['Unnamed: 0', 'Stock', 'Datetime', 'Open', 'High', 'Low', 'Close', 'Volume', 'Dividends', 'Stock Splits', 'Date', 'tweet_count', 'Likes', 'Retweets', 'Replies', 'Quotes', 'Positive_mean', 'Positive_std', 'Neutral_mean', 'Neutral_std', 'Negative_mean', 'Negative_std', 'tweet_count_7d_avg', 'tweet_count_30d_avg', 'Likes_7d_avg', 'Likes_30d_avg', 'Retweets_7d_avg', 'Retweets_30d_avg', 'Replies_7d_avg', 'Replies_30d_avg', 'Quotes_7d_avg', 'Quotes_30d_avg']

Data info:
   - Date range: 2018-01-01 00:00:00 to 2023-01-13 00:00:00
   - Stocks: ['ADANIENT' 'BHARTIARTL' 'HCLTECH' 'HDFCBANK' 'ICICIBANK' 'INFY' 'SBIN'
 'TATASTEEL']
   - Total samples: 9968

 Sample data:
   Unnamed: 0     Stock   Datetime       Open        High        Low  \
0           0  ADANIENT 2018-01-01  89.037562   91.212490  88.607948   
1           1  ADANIENT 2018-01-02  89.198668   89.896793  87.077448   
2           2  ADANIENT

In [8]:

# 2 FEATURE ENGINEERING


print("Starting feature engineering...")

# Create a copy for feature engineering
df_features = df.copy()

# Technical indicators for stock data
print("Adding technical indicators...")

# Price-based features
df_features['price_change'] = df_features['Close'].pct_change()
df_features['price_change_abs'] = df_features['price_change'].abs()
df_features['high_low_ratio'] = df_features['High'] / df_features['Low']
df_features['open_close_ratio'] = df_features['Open'] / df_features['Close']

# Moving averages
df_features['ma_5'] = df_features.groupby('Stock')['Close'].rolling(window=5).mean().reset_index(0, drop=True)
df_features['ma_10'] = df_features.groupby('Stock')['Close'].rolling(window=10).mean().reset_index(0, drop=True)
df_features['ma_20'] = df_features.groupby('Stock')['Close'].rolling(window=20).mean().reset_index(0, drop=True)

# Bollinger Bands
df_features['bb_upper'] = df_features.groupby('Stock')['Close'].rolling(window=20).mean().reset_index(0, drop=True) + \
                          (df_features.groupby('Stock')['Close'].rolling(window=20).std().reset_index(0, drop=True) * 2)
df_features['bb_lower'] = df_features.groupby('Stock')['Close'].rolling(window=20).mean().reset_index(0, drop=True) - \
                          (df_features.groupby('Stock')['Close'].rolling(window=20).std().reset_index(0, drop=True) * 2)
df_features['bb_position'] = (df_features['Close'] - df_features['bb_lower']) / (df_features['bb_upper'] - df_features['bb_lower'])

# RSI
def calculate_rsi(prices, window=14):
    delta = prices.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

df_features['rsi'] = df_features.groupby('Stock')['Close'].apply(calculate_rsi).reset_index(0, drop=True)

# Volume features
df_features['volume_ma_5'] = df_features.groupby('Stock')['Volume'].rolling(window=5).mean().reset_index(0, drop=True)
df_features['volume_ratio'] = df_features['Volume'] / df_features['volume_ma_5']

# Time-based features
df_features['day_of_week'] = df_features['Datetime'].dt.dayofweek
df_features['month'] = df_features['Datetime'].dt.month
df_features['quarter'] = df_features['Datetime'].dt.quarter
df_features['is_month_start'] = df_features['Datetime'].dt.is_month_start.astype(int)
df_features['is_month_end'] = df_features['Datetime'].dt.is_month_end.astype(int)

# Lag features for stock prices
df_features['close_lag_1'] = df_features.groupby('Stock')['Close'].shift(1)
df_features['close_lag_2'] = df_features.groupby('Stock')['Close'].shift(2)
df_features['close_lag_5'] = df_features.groupby('Stock')['Close'].shift(5)

# Lag features for Twitter metrics
twitter_cols = ['tweet_count', 'Likes', 'Retweets', 'Replies', 'Quotes']
for col in twitter_cols:
    if col in df_features.columns:
        df_features[f'{col}_lag_1'] = df_features.groupby('Stock')[col].shift(1)
        df_features[f'{col}_lag_2'] = df_features.groupby('Stock')[col].shift(2)

# Sentiment lag features
sentiment_cols = ['Positive_mean', 'Neutral_mean', 'Negative_mean']
for col in sentiment_cols:
    if col in df_features.columns:
        df_features[f'{col}_lag_1'] = df_features.groupby('Stock')[col].shift(1)
        df_features[f'{col}_lag_2'] = df_features.groupby('Stock')[col].shift(2)

print("Technical indicators and features added!")
print(f"New shape: {df_features.shape}")
print(f"New columns: {len(df_features.columns)}")

Starting feature engineering...
Adding technical indicators...
Technical indicators and features added!
New shape: (9968, 69)
New columns: 69


In [9]:

# DATA PREPARATION AND CLEANING


print("Preparing data for modeling...")

# Remove rows with NaN values (from lag features and technical indicators)
df_clean = df_features.dropna().reset_index(drop=True)

print(f"Data cleaned!")
print(f"Shape after cleaning: {df_clean.shape}")
print(f"Columns: {len(df_clean.columns)}")

# Check for infinite values
inf_cols = df_clean.columns[df_clean.isin([np.inf, -np.inf]).any()].tolist()
if inf_cols:
    print(f"Found infinite values in columns: {inf_cols}")
    # Replace infinite values with NaN and then forward fill
    df_clean = df_clean.replace([np.inf, -np.inf], np.nan)
    df_clean = df_clean.fillna(method='ffill')

# Define target variable (next day's close price)
df_clean['target'] = df_clean.groupby('Stock')['Close'].shift(-1)

# Remove the last row for each stock (no target available)
df_clean = df_clean.dropna(subset=['target']).reset_index(drop=True)

print(f"Target variable created!")
print(f"Final shape: {df_clean.shape}")

# Display target variable info
print(f"\nTarget variable info:")
print(f"   - Target range: {df_clean['target'].min():.2f} to {df_clean['target'].max():.2f}")
print(f"   - Target mean: {df_clean['target'].mean():.2f}")
print(f"   - Target std: {df_clean['target'].std():.2f}")

Preparing data for modeling...
Data cleaned!
Shape after cleaning: (9805, 69)
Columns: 69
Target variable created!
Final shape: (9797, 70)

Target variable info:
   - Target range: 10.75 to 4165.30
   - Target mean: 670.04
   - Target std: 579.62


In [10]:

# FEATURE SELECTION AND DATA SPLITTING

print("Selecting features for modeling...")

# Exclude non-feature columns
exclude_cols = ['Stock', 'Datetime', 'Date', 'target', 'Open', 'High', 'Low', 'Close', 'Volume']
feature_cols = [col for col in df_clean.columns if col not in exclude_cols]

print(f"Selected {len(feature_cols)} features for modeling")
print(f"Feature columns: {feature_cols[:10]}...")  # Show first 10 features

# Prepare X and y
X = df_clean[feature_cols]
y = df_clean['target']

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

# Check for any remaining NaN values
print(f"\nChecking for NaN values:")
print(f"   - X NaN count: {X.isnull().sum().sum()}")
print(f"   - y NaN count: {y.isnull().sum()}")

# Time-based train-test split (80% train, 20% test)
print(f"\nCreating time-based train-test split...")

# Sort by datetime to maintain temporal order
df_clean = df_clean.sort_values('Datetime').reset_index(drop=True)
split_idx = int(len(df_clean) * 0.8)

X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]
y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]

print(f"Train-test split created!")
print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"Training date range: {df_clean.iloc[:split_idx]['Datetime'].min()} to {df_clean.iloc[:split_idx]['Datetime'].max()}")
print(f"Test date range: {df_clean.iloc[split_idx:]['Datetime'].min()} to {df_clean.iloc[split_idx:]['Datetime'].max()}")

Selecting features for modeling...
Selected 61 features for modeling
Feature columns: ['Unnamed: 0', 'Dividends', 'Stock Splits', 'tweet_count', 'Likes', 'Retweets', 'Replies', 'Quotes', 'Positive_mean', 'Positive_std']...
X shape: (9797, 61)
y shape: (9797,)

Checking for NaN values:
   - X NaN count: 0
   - y NaN count: 0

Creating time-based train-test split...
Train-test split created!
Training set: 7837 samples
Test set: 1960 samples
Training date range: 2018-01-29 00:00:00 to 2022-01-18 00:00:00
Test date range: 2022-01-19 00:00:00 to 2023-01-12 00:00:00


In [12]:

# FEATURE SCALING

print("Scaling features...")

# Initialize scaler
scaler = StandardScaler()

# Fit scaler on training data only
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Features scaled!")
print(f"Training set scaled shape: {X_train_scaled.shape}")
print(f"Test set scaled shape: {X_test_scaled.shape}")

# Convert back to DataFrames for easier handling
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=feature_cols, index=X_train.index)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=feature_cols, index=X_test.index)

print(f"Scaled data converted to DataFrames")

Scaling features...
Features scaled!
Training set scaled shape: (7837, 61)
Test set scaled shape: (1960, 61)
Scaled data converted to DataFrames


In [14]:

# XGBOOST (WITH REGULARIZATION)

print("Training XGBoost with regularization...")

# Improved XGBoost parameters to reduce overfitting
xgb_improved_params = {
    'n_estimators': 100,
    'max_depth': 4,  # Reduced from 6
    'learning_rate': 0.05,  # Reduced from 0.1
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,  # L1 regularization
    'reg_lambda': 0.1,  # L2 regularization
    'random_state': 42,
    'n_jobs': -1
}

# Initialize and train improved XGBoost
xgb_improved = xgb.XGBRegressor(**xgb_improved_params)
xgb_improved.fit(X_train_scaled_df, y_train)

print(f"XGBoost model trained!")

# Make predictions
y_pred_xgb_imp_train = xgb_improved.predict(X_train_scaled_df)
y_pred_xgb_imp_test = xgb_improved.predict(X_test_scaled_df)

# Calculate metrics
train_rmse_xgb_imp = np.sqrt(mean_squared_error(y_train, y_pred_xgb_imp_train))
test_rmse_xgb_imp = np.sqrt(mean_squared_error(y_test, y_pred_xgb_imp_test))
train_r2_xgb_imp = r2_score(y_train, y_pred_xgb_imp_train)
test_r2_xgb_imp = r2_score(y_test, y_pred_xgb_imp_test)

print(f"\n XGBoost Performance:")
print(f"   - Train RMSE: {train_rmse_xgb_imp:.4f}")
print(f"   - Test RMSE: {test_rmse_xgb_imp:.4f}")
print(f"   - Train R²: {train_r2_xgb_imp:.4f}")
print(f"   - Test R²: {test_r2_xgb_imp:.4f}")

Training XGBoost with regularization...
XGBoost model trained!

 XGBoost Performance:
   - Train RMSE: 18.0610
   - Test RMSE: 53.0642
   - Train R²: 0.9990
   - Test R²: 0.9217


In [16]:

#  MODEL TRAINING - LIGHTGBM

print("Training LightGBM model...")

# Clean feature names - remove special characters that LightGBM doesn't like
def clean_feature_names(feature_names):
    """Clean feature names for LightGBM compatibility"""
    cleaned = []
    for name in feature_names:
        # Replace special characters with underscores
        cleaned_name = name.replace('(', '_').replace(')', '_').replace(' ', '_')
        cleaned_name = cleaned_name.replace('-', '_').replace('/', '_').replace('\\', '_')
        cleaned_name = cleaned_name.replace('[', '_').replace(']', '_').replace(':', '_')
        cleaned_name = cleaned_name.replace('+', '_').replace('*', '_').replace('=', '_')
        cleaned_name = cleaned_name.replace('&', '_').replace('%', '_').replace('$', '_')
        cleaned_name = cleaned_name.replace('#', '_').replace('@', '_').replace('!', '_')
        cleaned_name = cleaned_name.replace('?', '_').replace(';', '_').replace(',', '_')
        cleaned_name = cleaned_name.replace('.', '_').replace('|', '_').replace('~', '_')
        cleaned_name = cleaned_name.replace('`', '_').replace('"', '_').replace("'", '_')
        cleaned_name = cleaned_name.replace('<', '_').replace('>', '_').replace('{', '_')
        cleaned_name = cleaned_name.replace('}', '_').replace('^', '_').replace('=', '_')
        
        # Remove multiple consecutive underscores
        while '__' in cleaned_name:
            cleaned_name = cleaned_name.replace('__', '_')
        
        # Remove leading/trailing underscores
        cleaned_name = cleaned_name.strip('_')
        
        # Ensure it's not empty
        if not cleaned_name:
            cleaned_name = f"feature_{len(cleaned_names)}"
        
        cleaned.append(cleaned_name)
    
    return cleaned

# Clean feature names
cleaned_feature_names = clean_feature_names(feature_cols)
print(f"Feature names cleaned for LightGBM compatibility")

# Create new DataFrames with cleaned feature names
X_train_clean = X_train_scaled_df.copy()
X_test_clean = X_test_scaled_df.copy()
X_train_clean.columns = cleaned_feature_names
X_test_clean.columns = cleaned_feature_names

print(f"Cleaned feature names sample: {cleaned_feature_names[:5]}")

# LightGBM parameters with regularization
lgb_params = {
    'n_estimators': 100,
    'max_depth': 4,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,  # L1 regularization
    'reg_lambda': 0.1,  # L2 regularization
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

# Initialize and train LightGBM
lgb_model = lgb.LGBMRegressor(**lgb_params)
lgb_model.fit(X_train_clean, y_train)

print(f"LightGBM model trained!")

# Make predictions
y_pred_lgb_train = lgb_model.predict(X_train_clean)
y_pred_lgb_test = lgb_model.predict(X_test_clean)

# Calculate metrics
train_rmse_lgb = np.sqrt(mean_squared_error(y_train, y_pred_lgb_train))
test_rmse_lgb = np.sqrt(mean_squared_error(y_test, y_pred_lgb_test))
train_r2_lgb = r2_score(y_train, y_pred_lgb_train)
test_r2_lgb = r2_score(y_test, y_pred_lgb_test)

print(f"\nLightGBM Performance:")
print(f"   - Train RMSE: {train_rmse_lgb:.4f}")
print(f"   - Test RMSE: {test_rmse_lgb:.4f}")
print(f"   - Train R²: {train_r2_lgb:.4f}")
print(f"   - Test R²: {test_r2_lgb:.4f}")

Training LightGBM model...
Feature names cleaned for LightGBM compatibility
Cleaned feature names sample: ['Unnamed_0', 'Dividends', 'Stock_Splits', 'tweet_count', 'Likes']
LightGBM model trained!

LightGBM Performance:
   - Train RMSE: 18.3953
   - Test RMSE: 49.8725
   - Train R²: 0.9990
   - Test R²: 0.9308


In [19]:
# ================================================================
# QUICK LIGHTGBM OPTIMIZATION (BEST BANG FOR BUCK)
# ================================================================

print("🔧 Quick LightGBM optimization...")

from sklearn.model_selection import GridSearchCV

# Focus on the most important parameters only
param_grid = {
    'n_estimators': [100, 150],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.03, 0.05, 0.1],
    'reg_alpha': [0.01, 0.1],
    'reg_lambda': [0.01, 0.1]
}

# Initialize base model
base_lgb = lgb.LGBMRegressor(
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

# Quick grid search
print("⏳ Running quick grid search...")
grid_search = GridSearchCV(
    estimator=base_lgb,
    param_grid=param_grid,
    cv=3,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

# Fit and get best model
grid_search.fit(X_train_clean, y_train)
best_lgb = grid_search.best_estimator_

print(f"✅ Optimization completed!")
print(f" Best parameters: {grid_search.best_params_}")

# Test the improved model
y_pred_improved = best_lgb.predict(X_test_clean)
test_rmse_improved = np.sqrt(mean_squared_error(y_test, y_pred_improved))
test_r2_improved = r2_score(y_test, y_pred_improved)

print(f"\n📊 Improved LightGBM Performance:")
print(f"   - Test RMSE: {test_rmse_improved:.4f}")
print(f"   - Test R²: {test_r2_improved:.4f}")
print(f"   - Improvement: +{test_r2_improved - 0.9308:.4f} R²")

🔧 Quick LightGBM optimization...
⏳ Running quick grid search...
✅ Optimization completed!
 Best parameters: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 150, 'reg_alpha': 0.01, 'reg_lambda': 0.01}

📊 Improved LightGBM Performance:
   - Test RMSE: 44.1442
   - Test R²: 0.9458
   - Improvement: +0.0150 R²


In [ ]:
# Save the optimized model
print("Saving optimized model...")
import joblib

# Save best LightGBM model
model_path = os.path.join(models_dir, "optimized_lightgbm_model.joblib")
joblib.dump(best_lgb, model_path)
print(f"Optimized model saved to: {model_path}")

# Save the scaler
scaler_path = os.path.join(models_dir, "feature_scaler.joblib")
joblib.dump(scaler, scaler_path)
print(f"Feature scaler saved to: {scaler_path}")

print(f"\nModel optimization complete!")
print(f"Final performance: Test R² = {test_r2_improved:.4f}")
print(f"Models saved in: {models_dir}")

Saving optimized model...
✅ Optimized model saved to: C:\Users\PC\OneDrive\Documents\DS project\models\optimized_lightgbm_model.joblib
✅ Feature scaler saved to: C:\Users\PC\OneDrive\Documents\DS project\models\feature_scaler.joblib

�� Model optimization complete!
�� Final performance: Test R² = 0.9458
�� Models saved in: C:\Users\PC\OneDrive\Documents\DS project\models
